In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import fastf1
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Enable FastF1 caching
import os
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

### 2025 Data

In [3]:
# Load 2025 Italian GP race session, lap and sector times
session_2025 = fastf1.get_session(2026, "Italy", "R")
session_2025.load()
laps_2025 = session_2025.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
laps_2025.dropna(inplace=True)

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 12 completed the race distance 00:00.013000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '3', '1', '81', '44', '10', '41', '43', '22', '5', '27

In [4]:
# Convert times to seconds
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2025[f"{col} (s)"] = laps_2025[col].dt.total_seconds()

In [5]:
# Group by driver to get average sector times per driver
sector_times_2025 = laps_2025.groupby("Driver")[["Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)"]].mean().reset_index()

### Quali Data 2026

In [6]:
# Qualifying data for 2026 Italian GP (Monza)
# Time = lap from the last session each driver reached (Q3 for P1-10, Q2 for P11-16, Q1 for P17-22)
qualifying_2026 = pd.DataFrame({
    "Driver": ["GAS", "RUS", "PIA", "LEC", "HAM", "VER", "ANT", "COL", "NOR", "LIN",
               "BOR", "BEA", "HUL", "LAW", "SAI", "OCO",
               "TSU", "ALB", "BOT", "PER", "ALO", "STR"],
    "QualifyingTime (s)": [81.786, 81.846, 81.966, 82.004, 82.011, 82.070, 82.093, 82.220, 82.256, 82.286,
                           82.517, 82.756, 82.779, 82.821, 83.453, 83.454,
                           83.755, 84.356, 84.364, 84.595, 85.150, 85.222]
})

### Weather component

In [7]:
API_KEY = "put your API here"
# Monza (Autodromo Nazionale Monza), Italy -- 2026 Italian GP venue
weather_url = f"http://api.openweathermap.org/data/2.5/forecast?lat=45.6156&lon=9.2811&appid={API_KEY}&units=metric"
response = requests.get(weather_url)
weather_data = response.json()
print(weather_data)

{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1791936000, 'main': {'temp': 14.43, 'feels_like': 14.1, 'temp_min': 14.34, 'temp_max': 14.43, 'pressure': 1026, 'sea_level': 1026, 'grnd_level': 1002, 'humidity': 83, 'temp_kf': 0.09, 'dew_point': 11.58}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03n'}], 'clouds': {'all': 32}, 'wind': {'speed': 1.23, 'deg': 46, 'gust': 2.47}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2026-10-14 00:00:00'}, {'dt': 1791946800, 'main': {'temp': 13.53, 'feels_like': 13.18, 'temp_min': 13.06, 'temp_max': 13.53, 'pressure': 1026, 'sea_level': 1026, 'grnd_level': 1002, 'humidity': 86, 'temp_kf': 0.47, 'dew_point': 11.24}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03n'}], 'clouds': {'all': 27}, 'wind': {'speed': 1.52, 'deg': 76, 'gust': 3.04}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2026-10-14 03:00:00'}, {'dt': 1791957600, '

In [8]:
# Extract the relevant weather data for the race (Sunday at 3pm local time)
forecast_time = "2026-09-06 15:00:00"
forecast_data = None
if "list" in weather_data:
    for forecast in weather_data["list"]:
        if forecast["dt_txt"] == forecast_time:
            forecast_data = forecast
            break
else:
    print(f"Weather API call failed ({weather_data.get('message', weather_data)}); using default values.")

if forecast_data:
    rain_probability = forecast_data["pop"]
    temperature = forecast_data["main"]["temp"]
else:
    rain_probability = 0
    temperature = 20

In [9]:
# Merge qualifying data with sector times data
merged_data = qualifying_2026.merge(sector_times_2025, left_on="Driver", right_on="Driver", how="left")

In [10]:
# Create weather features for the model
merged_data["RainProbability"] = rain_probability
merged_data["Temperature"] = temperature

In [11]:
# Attach average lap time to merged_data the same way sector times were attached (left join),
# so a driver with no clean lap (e.g. retired, like LEC) is dropped consistently from X and y.
merged_data["LapTime (s)"] = merged_data["Driver"].map(laps_2025.groupby("Driver")["LapTime (s)"].mean())
merged_data.dropna(subset=["LapTime (s)"], inplace=True)

# Define feature set (Qualifying + Sector Times + Weather)
X = merged_data[["QualifyingTime (s)", "Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)", "RainProbability", "Temperature"]].fillna(0)

# Use average lap time as the target
y = merged_data["LapTime (s)"]

## Train Gradient Boosting Model

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=38)
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=38)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",38
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf

In [13]:
# Predict race times using 2026 qualifying and sector data
predicted_race_times = model.predict(X)
merged_data["PredictedRaceTime (s)"] = predicted_race_times
race_predictions_2026 = merged_data.sort_values(by="PredictedRaceTime (s)")

In [14]:
# Evaluate
y_pred = model.predict(X_test)
print(f"\n Model Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} seconds")


 Model Error (MAE): 0.28 seconds


## Random Forest Regressor

In [ ]:
# Tune Random Forest hyperparameters with 5-fold grid search
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}
rf_grid_search = GridSearchCV(
    RandomForestRegressor(random_state=38),
    rf_param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
rf_grid_search.fit(X_train, y_train)
rf = rf_grid_search.best_estimator_
print(f"Best RF params: {rf_grid_search.best_params_}")

rf_y_pred = rf.predict(X_test)
print(f"\n Model Error (MAE): {mean_absolute_error(y_test, rf_y_pred):.2f} seconds")

# Predict race times for all drivers using the tuned Random Forest model
merged_data["PredictedRaceTime_RF (s)"] = rf.predict(X)

### Predicted vs Actual Result (2026 Italian GP)

In [16]:
# Actual 2026 Italian GP race result (formula1.com/en/results/2026/races/1293/italy/race-result)
actual_results_2026 = pd.DataFrame({
    "Driver": ["ANT", "RUS", "VER", "NOR", "PIA", "HAM", "GAS", "LIN", "COL", "TSU",
               "BOR", "HUL", "SAI", "LAW", "BEA", "OCO", "ALB", "PER", "BOT",
               "STR", "ALO", "LEC"],
    "ActualPosition": list(range(1, 23)),
    "Status": ["Finished"] * 19 + ["DNF (26 laps)", "DNF (23 laps)", "DNF (1 lap)"]
})

## Positions and Lap times prediction using RF

In [18]:
# Predicted finishing order (Random Forest ranking): rank drivers by predicted average lap time, fastest first.
# ActualAvgLapTime (s) comes from the real race laps loaded via FastF1 (merged_data),
# not the results page, since lapped cars there only show "+1 lap" / "+2 laps", not a time.
predicted_ranking = merged_data.sort_values("PredictedRaceTime_RF (s)")[
    ["Driver", "LapTime (s)", "PredictedRaceTime_RF (s)"]
].reset_index(drop=True)
predicted_ranking = predicted_ranking.rename(columns={"LapTime (s)": "ActualAvgLapTime (s)"})
predicted_ranking["PredictedPosition"] = predicted_ranking.index + 1
predicted_ranking["TimeDiff (s)"] = predicted_ranking["ActualAvgLapTime (s)"] - predicted_ranking["PredictedRaceTime_RF (s)"]

# Combine predicted and actual results. Left join on the actual results keeps LEC
# in the table even though the model couldn't predict him (no clean lap data to
# train/predict on after his lap-1 DNF) -- he just gets NaN predicted/actual values.
results_comparison = actual_results_2026.merge(predicted_ranking, on="Driver", how="left")
results_comparison["PositionDiff"] = results_comparison["ActualPosition"] - results_comparison["PredictedPosition"]

# Order from P1 down to last classified position
results_comparison = results_comparison[
    ["Driver", "ActualPosition", "PredictedPosition", "PositionDiff",
     "ActualAvgLapTime (s)", "PredictedRaceTime_RF (s)", "TimeDiff (s)", "Status"]
].sort_values("ActualPosition").reset_index(drop=True)
results_comparison

,Driver,ActualPosition,PredictedPosition,PositionDiff,ActualAvgLapTime (s),PredictedRaceTime_RF (s),TimeDiff (s),Status
0,ANT,1,9.0,-8.0,88.502720,88.408448,0.094272,Finished
1,RUS,2,1.0,1.0,87.052020,87.293395,-0.241374,Finished
2,VER,3,2.0,1.0,87.391429,87.551269,-0.159841,Finished
3,NOR,4,6.0,-2.0,87.618224,87.746824,-0.128599,Finished
4,PIA,5,4.0,1.0,87.617571,87.654640,-0.037068,Finished
5,HAM,6,5.0,1.0,87.810612,87.726104,0.084508,Finished
6,GAS,7,3.0,4.0,87.620102,87.638383,-0.018281,Finished
7,LIN,8,8.0,0.0,88.175122,88.325352,-0.150229,Finished
8,COL,9,7.0,2.0,87.930490,88.208809,-0.278319,Finished
9,TSU,10,11.0,-1.0,88.431816,88.798346,-0.366530,Finished


In [19]:
# Average position error (LEC excluded -- no prediction was made for him)
mean_abs_position_error = results_comparison["PositionDiff"].abs().mean()
print(f"Mean Absolute Position Error: {mean_abs_position_error:.2f} positions")

Mean Absolute Position Error: 2.10 positions


## Lap Times up close

In [20]:
# Lap time comparison: actual vs. Random Forest predicted, per driver
lap_time_comparison = merged_data[["Driver", "LapTime (s)", "PredictedRaceTime_RF (s)"]].copy()
lap_time_comparison = lap_time_comparison.rename(columns={"LapTime (s)": "ActualAvgLapTime (s)"})
lap_time_comparison["TimeDiff (s)"] = lap_time_comparison["ActualAvgLapTime (s)"] - lap_time_comparison["PredictedRaceTime_RF (s)"]
lap_time_comparison

,Driver,ActualAvgLapTime (s),PredictedRaceTime_RF (s),TimeDiff (s)
0,GAS,87.620102,87.638383,-0.018281
1,RUS,87.052020,87.293395,-0.241374
2,PIA,87.617571,87.654640,-0.037068
4,HAM,87.810612,87.726104,0.084508
5,VER,87.391429,87.551269,-0.159841
6,ANT,88.502720,88.408448,0.094272
7,COL,87.930490,88.208809,-0.278319
8,NOR,87.618224,87.746824,-0.128599
9,LIN,88.175122,88.325352,-0.150229
10,BOR,89.880400,89.648084,0.232316


Random Forest Regressor scored better than Gradient Boost Regressor by 0.09 seconds MAE, making it better model. Prediction on final position varies, but looking at lap times RF predicts well. Potential improvement is tuning models and comparing. Error of 0.19 s is significant if we consider this is Formula 1.

## RF Tuning

In [25]:
param_grid = {
    "n_estimators": [100, 200, 300, 600],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": np.linspace(0.01, 1.0, 10, endpoint=True),
    "min_samples_leaf": np.linspace(0.1, 0.5, 5, endpoint=True),
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=38),
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_test)

print(f"Best params: {grid_search.best_params_}")
print(f"Test MAE: {mean_absolute_error(y_test, best_rf_pred):.2f} seconds")

Best params: {'max_depth': None, 'min_samples_leaf': np.float64(0.1), 'min_samples_split': np.float64(0.1), 'n_estimators': 100}
Test MAE: 0.20 seconds
